# 02 · Preprocessing
Derive anomaly labels from RUL, drop constant sensors, recover operating
conditions with Spark MLlib k-means, and standardize each sensor within its
condition.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # project root, so `import src` works

In [2]:
from src.spark_utils import get_spark
from src.ingest import ingest
from src.preprocess import preprocess
from src.config import W_LABEL
spark = get_spark()

In [3]:
df = ingest(spark, 'FD001')
df, keep = preprocess(df, 'FD001')
print('kept (informative) sensors:', keep)

kept (informative) sensors: ['s_2', 's_3', 's_4', 's_6', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21']


Labels: a cycle is anomalous if RUL ≤ 30 cycles before failure. Class balance:

In [4]:
df.groupBy('label').count().show()

+-----+-----+
|label|count|
+-----+-----+
|    1| 3100|
|    0|17531|
+-----+-----+



A few standardized rows with their RUL/label and partition keys:

In [5]:
df.select('unit_nr','time_cycles','RUL','label','cond','row_id','eng_part','rand_part')\
  .show(8)

+-------+-----------+---+-----+----+------+--------+---------+
|unit_nr|time_cycles|RUL|label|cond|row_id|eng_part|rand_part|
+-------+-----------+---+-----+----+------+--------+---------+
|      1|          1|191|    0|   0|100001|       1|        3|
|      1|          2|190|    0|   0|100002|       1|        0|
|      1|          3|189|    0|   0|100003|       1|        2|
|      1|          4|188|    0|   0|100004|       1|        0|
|      1|          5|187|    0|   0|100005|       1|        0|
|      1|          6|186|    0|   0|100006|       1|        1|
|      1|          7|185|    0|   0|100007|       1|        2|
|      1|          8|184|    0|   0|100008|       1|        0|
+-------+-----------+---+-----+----+------+--------+---------+
only showing top 8 rows


In [6]:
spark.stop()